In [1]:
!pip install -q transformers datasets accelerate peft bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 39.8 MB/s eta 0:00:00


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "Qwen/Qwen3-4B"  # closest widely available

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,  # ✅ bf16
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [7]:
from datasets import load_dataset

dataset = load_dataset("databricks/databricks-dolly-15k")

def format_prompt(example):
    if example["context"]:
        return f"""Instruction: {example['instruction']}

Context: {example['context']}

Response: {example['response']}"""
    else:
        return f"""Instruction: {example['instruction']}

Response: {example['response']}"""

dataset = dataset["train"].map(lambda x: {"text": format_prompt(x)})
dataset = dataset.shuffle(seed=42).select(range(3000))  # subset for T4

In [8]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],  # Qwen attention layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 11,796,480 || all params: 4,034,264,576 || trainable%: 0.2924


In [9]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./qwen-dolly-peft",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=50,
    save_steps=200,
    bf16=True,        # ✅ use bf16
    fp16=False,       # ❌ disable fp16
    optim="paged_adamw_8bit",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    processing_class=tokenizer,   # ✅ new replacement
    formatting_func=lambda x: x["text"],
)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss
50,1.987253
100,1.971516
150,1.840653
200,1.911576
250,1.843994
300,1.924190
350,1.898650


TrainOutput(global_step=375, training_loss=1.912470937093099, metrics={'train_runtime': 353.4015, 'train_samples_per_second': 8.489, 'train_steps_per_second': 1.061, 'total_flos': 2.412220144983245e+16, 'train_loss': 1.912470937093099})

In [10]:
trainer.model.save_pretrained("./qwen-dolly-peft")
tokenizer.save_pretrained("./qwen-dolly-peft")

('./qwen-dolly-peft/tokenizer_config.json',
 './qwen-dolly-peft/chat_template.jinja',
 './qwen-dolly-peft/tokenizer.json')

In [11]:
from peft import PeftModel

# reload base + adapter
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = PeftModel.from_pretrained(base_model, "./qwen-dolly-peft")

prompt = """### Instruction:
Explain what is machine learning in simple terms.

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=150,
    temperature=0.7,
    top_p=0.9
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

### Instruction:
Explain what is machine learning in simple terms.

### Response:
Machine learning is the science of making machines learn from data. It is the ability to learn from experience. Machine learning is a subset of AI. The difference between AI and machine learning is that AI is a much broader field. AI includes all the techniques that make machines smart, while machine learning is a subset of AI that focuses on learning from data.


In [12]:
import shutil

# zip the folder
shutil.make_archive("my_folder", 'zip', "/content/qwen-dolly-peft")

# download
from google.colab import files
files.download("my_folder.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>